# ClipCap caption generation pipeline

Notebook này xây toàn bộ luồng inference của ClipCap từ một ảnh mới đến caption. Mục tiêu là tách từng công đoạn để có thể kiểm tra tensor, hiểu vai trò của từng model và phát hiện lỗi dễ hơn trước khi chuyển logic sang file `.py`. Phiên bản này không dùng hard prompt: visual prefix trực tiếp điều kiện hóa GPT-2 giống cách model đã được training.

Luồng chính:

```text
Image -> CLIPProcessor -> CLIP image feature -> TransformerMapper
      -> visual prefix -> GPT-2 beam search -> 5 candidates
      -> CLIP text-image reranking -> 1 final caption
```

Notebook dùng `final.pt` của giao thức `fixed_epoch` để inference. `best.pt` chỉ phục vụ phân tích theo validation loss, còn `latest.pt` dành chủ yếu cho việc tiếp tục training vì chứa cả optimizer state và random state.

## 1. Cấu trúc checkpoint

Trainer hiện tại lưu mỗi experiment theo cấu trúc sau:

```text
outputs/clipcap/
  train_1pct/
    seed_42/
      best.pt
      latest.pt
      final.pt
      config.json
      history.json
      result.json
  train_5pct/seed_42/
  train_10pct/seed_42/
  train_25pct/seed_42/
  train_100pct/seed_42/
```

Một bộ sáu file phải nằm chung trong đúng thư mục `subset/seed`. Giá trị `subset_name`, `seed` và `training_policy` trong `config.json` cho biết bộ checkpoint thuộc experiment nào. Đánh giá fixed-epoch luôn dùng trọng số Mapper trong `final.pt`.

In [ ]:
from __future__ import annotations

import json
import sys
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Any

import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from IPython.display import display
from transformers import (
    AutoTokenizer,
    CLIPModel,
    CLIPProcessor,
    GPT2Config,
    GPT2LMHeadModel,
)


def find_project_root(start: Path) -> Path:
    candidates = (start, *start.parents)
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("Không tìm thấy project root chứa src/ và requirements.txt")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clipcap.models.clipcap_model import ClipCaptionModel
from src.clipcap.models.mapping_network import TransformerMapper
from src.config.clipcap_config import CLIP_MODEL_NAME, CLIPCAP_OUTPUT_ROOT

print(f"Project root: {PROJECT_ROOT}")
print(f"Torch version: {torch.__version__}")

## 2. Chọn checkpoint, ảnh và thiết bị

Chỉnh ba biến `SUBSET_NAME`, `SEED` và `IMAGE_PATH` trước khi chạy toàn bộ notebook. Notebook kiểm tra một subset tại một thời điểm; pipeline `.py` sẽ chạy lần lượt cả năm subset trên cùng tập đánh giá bên ngoài.

`MAX_NEW_TOKENS` chỉ tính số token caption mới, không tính các visual-prefix token. Giá trị 15 được dùng để đồng nhất ngân sách generation với ZeroCap. `BEAM_SIZE = 5` giữ năm giả thuyết, và `NUM_RETURN_SEQUENCES = 5` trả cả năm cho CLIP reranking.

In [ ]:
SUBSET_NAME = "train_100pct"
SEED = 42
CHECKPOINT_DIR = Path(CLIPCAP_OUTPUT_ROOT) / SUBSET_NAME / f"seed_{SEED}"
CHECKPOINT_PATH = CHECKPOINT_DIR / "final.pt"

IMAGE_PATH = PROJECT_ROOT / "data" / "sample.jpg"
MAX_NEW_TOKENS = 15
BEAM_SIZE = 5
NUM_RETURN_SEQUENCES = 5
LENGTH_PENALTY = 1.0
EARLY_STOPPING = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Image path: {IMAGE_PATH}")
print(f"Device: {DEVICE}")

## 3. Kiểm tra bộ artifact

Inference fixed-epoch dùng trực tiếp `final.pt`, nhưng kiểm tra đủ sáu file giúp bảo đảm bạn đã đặt đúng một experiment hoàn chỉnh. Không nên trộn checkpoint của subset này với `config.json` của subset khác.

In [ ]:
REQUIRED_ARTIFACTS = (
    "best.pt",
    "latest.pt",
    "final.pt",
    "config.json",
    "history.json",
    "result.json",
)


def validate_artifact_directory(checkpoint_dir: Path) -> dict[str, Path]:
    checkpoint_dir = Path(checkpoint_dir)
    artifacts = {name: checkpoint_dir / name for name in REQUIRED_ARTIFACTS}
    missing = [name for name, path in artifacts.items() if not path.is_file()]
    if missing:
        missing_text = ", ".join(missing)
        raise FileNotFoundError(
            f"Thiếu artifact trong {checkpoint_dir}: {missing_text}. "
            "Hãy chép đúng bộ checkpoint đã sao lưu vào thư mục này."
        )
    return artifacts


artifacts = validate_artifact_directory(CHECKPOINT_DIR)
for name, path in artifacts.items():
    print(f"{name:12s} {path.stat().st_size / (1024 ** 2):9.2f} MB")

## 4. Đọc metadata và checkpoint

`final.pt` chứa trọng số Mapper sau epoch cuối của giao thức `fixed_epoch`. Đây là checkpoint chính thức để so sánh các subset trong cùng điều kiện số epoch. GPT-2 không nằm trong checkpoint vì GPT-2 bị đóng băng trong training; notebook sẽ tải lại đúng pretrained GPT-2 từ tên model đã lưu.

Chỉ load checkpoint do bạn tự tạo hoặc tin tưởng. File PyTorch có thể chứa dữ liệu pickle.

In [ ]:
with artifacts["config.json"].open("r", encoding="utf-8") as file:
    saved_config = json.load(file)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

if checkpoint.get("checkpoint_version") != 1:
    raise ValueError("Notebook chỉ hỗ trợ checkpoint_version=1")
if checkpoint.get("checkpoint_type") != "final":
    raise ValueError("CHECKPOINT_PATH phải trỏ tới final.pt")
if checkpoint.get("config") != saved_config:
    raise ValueError("config.json không khớp với config nằm trong final.pt")
if saved_config.get("subset_name") != SUBSET_NAME:
    raise ValueError("SUBSET_NAME không khớp với subset_name trong config.json")
if saved_config.get("seed") != SEED:
    raise ValueError("SEED không khớp với seed trong config.json")
if saved_config.get("training_policy") != "fixed_epoch":
    raise ValueError("Notebook inference này yêu cầu training_policy=fixed_epoch")
state = checkpoint.get("state", {})
final_epoch = state.get("last_epoch")
configured_epochs = saved_config.get("max_epochs")
if final_epoch != configured_epochs:
    raise ValueError("final.pt không chứa trọng số của epoch cuối theo config")

print(json.dumps(saved_config, ensure_ascii=False, indent=2))
print(f"Final epoch: {final_epoch}")
print(f"Best epoch (tham khảo): {state['best_epoch']}")
print(f"Best validation loss (tham khảo): {state['best_val_loss']:.6f}")

## 5. Tải CLIP, tokenizer và GPT-2

Ba thành phần có nhiệm vụ khác nhau:

- `CLIPProcessor`: resize, crop và normalize pixel đúng chuẩn pretrained CLIP.
- `CLIPModel`: biến ảnh thành một vector semantic. Với cấu hình dự án hiện tại vector có 512 chiều.
- `GPT-2`: nhận visual prefix trong không gian embedding 768 chiều và dự đoán token caption kế tiếp.

Lần chạy đầu có thể cần tải model từ Hugging Face. Những lần sau model được lấy từ cache cục bộ.

In [ ]:
gpt2_model_name = checkpoint.get("gpt2_model_name")
if not gpt2_model_name:
    raise KeyError("Checkpoint không chứa gpt2_model_name")

clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_encoder = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(gpt2_model_name, use_fast=True)
gpt2 = GPT2LMHeadModel.from_pretrained(gpt2_model_name).to(DEVICE)

if tokenizer.eos_token_id is None:
    raise ValueError("Tokenizer phải có eos_token_id để biết lúc nào dừng")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

for component in (clip_encoder, gpt2):
    for parameter in component.parameters():
        parameter.requires_grad = False
    component.eval()

print(f"CLIP model: {CLIP_MODEL_NAME}")
print(f"GPT-2 model: {gpt2_model_name}")
print(f"GPT-2 embedding dim: {gpt2.get_input_embeddings().embedding_dim}")

## 6. Dựng lại ClipCap và nạp Mapper

Checkpoint chỉ lưu `mapper_state_dict`, do đó phải dựng Mapper có kiến trúc giống hệt lúc training trước khi nạp trọng số. Các giá trị `clip_length`, `prefix_length`, số layer, số head, feed-forward dimension và dropout đều lấy từ `config.json`.

`clip_dim` được suy ra trực tiếp từ ma trận projection trong checkpoint. Cách này tránh hard-code 512 và phát hiện ngay nếu CLIP encoder không tương thích.

In [ ]:
mapper_state = checkpoint.get("mapper_state_dict")
if not isinstance(mapper_state, dict):
    raise TypeError("Checkpoint không chứa mapper_state_dict hợp lệ")

projection_key = "clip_projection.projection.weight"
if projection_key not in mapper_state:
    raise KeyError(f"Mapper state thiếu tensor {projection_key}")

clip_dim = int(mapper_state[projection_key].shape[1])
gpt_embedding_dim = int(gpt2.get_input_embeddings().embedding_dim)
clip_projection_dim = int(clip_encoder.config.projection_dim)
if clip_projection_dim != clip_dim:
    raise ValueError(
        f"CLIP output dim ({clip_projection_dim}) không khớp checkpoint ({clip_dim})"
    )

mapper = TransformerMapper(
    clip_dim=clip_dim,
    embedding_dim=gpt_embedding_dim,
    clip_length=int(saved_config["clip_length"]),
    prefix_length=int(saved_config["prefix_length"]),
    num_layers=int(saved_config["num_layers"]),
    num_heads=int(saved_config["num_heads"]),
    feedforward_dim=saved_config.get("feedforward_dim"),
    dropout=float(saved_config["dropout"]),
).to(DEVICE)
mapper.load_state_dict(mapper_state, strict=True)
mapper.eval()

clipcap_model = ClipCaptionModel(mapper=mapper, gpt2=gpt2).to(DEVICE)
clipcap_model.eval()

print(f"CLIP feature shape: [B, {clip_dim}]")
print(f"Visual prefix shape: [B, {mapper.prefix_length}, {mapper.embedding_dim}]")
print(mapper.count_parameters())

## 7. Mở ảnh và mã hóa bằng CLIP

Ảnh được chuyển sang RGB để loại bỏ khác biệt giữa ảnh grayscale, RGBA và RGB. Sau đó `CLIPProcessor` tạo `pixel_values`; CLIP image encoder biến chúng thành `image_features` có shape `[1, clip_dim]`.

Feature gốc từ `get_image_features` được giữ nguyên khi đưa vào Mapper vì training cũng dùng feature chưa chuẩn hóa L2. Đến bước CLIP reranking, notebook tạo bản sao đã chuẩn hóa để tính cosine similarity; tensor gốc không bị sửa.

In [ ]:
def load_rgb_image(image_path: str | Path) -> Image.Image:
    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy ảnh: {image_path}")
    with Image.open(image_path) as image:
        return image.convert("RGB")


@torch.inference_mode()
def encode_image_with_clip(
    image: Image.Image,
    processor: CLIPProcessor,
    encoder: CLIPModel,
    device: torch.device,
) -> torch.Tensor:
    processed = processor(images=image, return_tensors="pt")
    if "pixel_values" not in processed:
        raise KeyError("CLIPProcessor không trả về pixel_values")

    pixel_values = processed["pixel_values"].to(device)
    raw_features = encoder.get_image_features(pixel_values=pixel_values)
    image_features = (
        raw_features.pooler_output
        if hasattr(raw_features, "pooler_output")
        else raw_features
    )

    if image_features.ndim != 2 or image_features.size(0) != 1:
        raise ValueError("CLIP feature phải có shape [1, clip_dim]")
    if not torch.isfinite(image_features).all():
        raise ValueError("CLIP feature chứa NaN hoặc Inf")
    return image_features


image = load_rgb_image(IMAGE_PATH)
display(image)
image_features = encode_image_with_clip(image, clip_processor, clip_encoder, DEVICE)
print(f"Image feature shape: {tuple(image_features.shape)}")

## 8. Biến CLIP feature thành visual prefix

Mapper thực hiện ba bước nội bộ:

1. `ClipProjection` chiếu vector CLIP sang một chuỗi `clip_length` image token trong không gian embedding của GPT-2.
2. `PrefixTransformerEncoder` cho image token tương tác với các prefix query có thể học.
3. Mapper lấy `prefix_length` vị trí cuối làm visual prefix.

Visual prefix không phải token ID. Nó là embedding liên tục mà GPT-2 đọc như phần ngữ cảnh đứng trước caption.

In [ ]:
@torch.inference_mode()
def build_visual_prefix(
    image_features: torch.Tensor,
    model: ClipCaptionModel,
) -> torch.Tensor:
    mapper_parameter = next(model.mapper.parameters())
    features = image_features.to(
        device=mapper_parameter.device,
        dtype=mapper_parameter.dtype,
    )
    prefix = model.mapper(features)
    expected_shape = (
        features.size(0),
        model.mapper.prefix_length,
        model.mapper.embedding_dim,
    )
    if tuple(prefix.shape) != expected_shape:
        raise ValueError(
            f"Visual prefix có shape {tuple(prefix.shape)}, cần {expected_shape}"
        )
    return prefix


visual_prefix = build_visual_prefix(image_features, clipcap_model)
print(f"Visual prefix shape: {tuple(visual_prefix.shape)}")

## 9. Sinh 5 ứng viên bằng beam search và CLIP reranking

GPT-2 nhận trực tiếp visual prefix qua `inputs_embeds`; notebook không chèn hard prompt. Beam search giữ năm chuỗi có xác suất tổng thể tốt nhất thay vì chốt token tốt nhất ngay ở từng bước như greedy decoding. `generate()` quản lý KV cache và việc sắp xếp lại cache giữa các beam.

Sau khi decode năm beam, CLIP text encoder mã hóa đồng thời năm caption. Notebook chuẩn hóa L2 text feature và một bản sao của image feature, tính cosine similarity, rồi chọn caption có CLIP score lớn nhất. Beam score chỉ phản ánh độ trôi chảy theo GPT-2; quyết định cuối cùng chỉ dùng CLIP score để đồng nhất với giao thức ZeroCap.

Năm beam là năm hypothesis, nhưng beam search không bảo đảm năm chuỗi sau decode luôn khác nhau hoàn toàn. Caption rỗng bị gán CLIP score âm vô cùng để không thắng reranking khi còn ứng viên hợp lệ.

In [ ]:
@dataclass(frozen=True)
class BeamCandidate:
    beam_rank: int
    caption: str
    token_ids: tuple[int, ...]
    beam_score: float
    clip_score: float | None = None


@dataclass(frozen=True)
class GenerationResult:
    caption: str
    selected_beam_rank: int
    candidates: tuple[BeamCandidate, ...]


def _trim_at_eos(token_ids: list[int], eos_token_id: int) -> tuple[int, ...]:
    trimmed: list[int] = []
    for token_id in token_ids:
        trimmed.append(token_id)
        if token_id == eos_token_id:
            break
    return tuple(trimmed)


@torch.inference_mode()
def beam_search_from_prefix(
    gpt2_model: GPT2LMHeadModel,
    text_tokenizer: Any,
    prefix_embeddings: torch.Tensor,
    max_new_tokens: int,
    num_beams: int = 5,
    num_return_sequences: int = 5,
    length_penalty: float = 1.0,
    early_stopping: bool = True,
) -> tuple[BeamCandidate, ...]:
    integer_parameters = {
        "max_new_tokens": max_new_tokens,
        "num_beams": num_beams,
        "num_return_sequences": num_return_sequences,
    }
    for name, value in integer_parameters.items():
        if isinstance(value, bool) or not isinstance(value, int) or value < 1:
            raise ValueError(f"{name} phải là số nguyên dương")
    if num_beams < 2:
        raise ValueError("num_beams phải lớn hơn 1 để kích hoạt beam search")
    if num_return_sequences > num_beams:
        raise ValueError("num_return_sequences không được lớn hơn num_beams")
    if not isinstance(length_penalty, (int, float)) or isinstance(length_penalty, bool):
        raise TypeError("length_penalty phải là số")
    if prefix_embeddings.ndim != 3 or prefix_embeddings.size(0) != 1:
        raise ValueError("Notebook hiện hỗ trợ prefix shape [1, P, D]")

    eos_token_id = text_tokenizer.eos_token_id
    pad_token_id = text_tokenizer.pad_token_id
    if eos_token_id is None or pad_token_id is None:
        raise ValueError("Tokenizer phải có eos_token_id và pad_token_id")

    prefix_length = int(prefix_embeddings.size(1))
    max_positions = getattr(gpt2_model.config, "n_positions", None)
    if max_positions is None:
        max_positions = getattr(gpt2_model.config, "max_position_embeddings", None)
    if max_positions is not None and prefix_length + max_new_tokens > max_positions:
        raise ValueError(
            "Visual prefix cộng max_new_tokens vượt giới hạn vị trí của GPT-2"
        )

    attention_mask = torch.ones(
        (1, prefix_length),
        dtype=torch.long,
        device=prefix_embeddings.device,
    )
    generated = gpt2_model.generate(
        inputs_embeds=prefix_embeddings,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=num_beams,
        num_return_sequences=num_return_sequences,
        length_penalty=float(length_penalty),
        early_stopping=early_stopping,
        eos_token_id=eos_token_id,
        pad_token_id=pad_token_id,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True,
    )
    if generated.sequences.size(0) != num_return_sequences:
        raise ValueError("Beam search không trả về đúng số lượng sequence yêu cầu")
    if generated.sequences_scores is None:
        raise ValueError("Beam search không trả về sequence score")

    raw_token_ids = generated.sequences.detach().cpu().tolist()
    captions = text_tokenizer.batch_decode(
        generated.sequences,
        skip_special_tokens=True,
    )
    candidates = []
    for index, (ids, caption, score) in enumerate(
        zip(raw_token_ids, captions, generated.sequences_scores),
        start=1,
    ):
        candidates.append(
            BeamCandidate(
                beam_rank=index,
                caption=caption.strip(),
                token_ids=_trim_at_eos(ids, eos_token_id),
                beam_score=float(score.detach().cpu()),
            )
        )
    return tuple(candidates)


def compute_clip_similarity_scores(
    image_features: torch.Tensor,
    text_features: torch.Tensor,
) -> torch.Tensor:
    if image_features.ndim != 2 or image_features.size(0) != 1:
        raise ValueError("image_features phải có shape [1, clip_dim]")
    if text_features.ndim != 2 or text_features.size(0) < 1:
        raise ValueError("text_features phải có shape [N, clip_dim]")
    if image_features.size(1) != text_features.size(1):
        raise ValueError("Image và text feature phải có cùng chiều embedding")
    if not torch.isfinite(image_features).all() or not torch.isfinite(text_features).all():
        raise ValueError("CLIP feature chứa NaN hoặc Inf")
    if image_features.norm(dim=-1).min() <= 0 or text_features.norm(dim=-1).min() <= 0:
        raise ValueError("Không thể normalize CLIP feature có norm bằng 0")

    normalized_image = F.normalize(image_features.float(), dim=-1)
    normalized_text = F.normalize(text_features.float(), dim=-1)
    return normalized_text @ normalized_image.transpose(0, 1)


@torch.inference_mode()
def clip_rerank_candidates(
    candidates: tuple[BeamCandidate, ...],
    image_features: torch.Tensor,
    processor: CLIPProcessor,
    encoder: CLIPModel,
    device: torch.device,
) -> GenerationResult:
    if not candidates:
        raise ValueError("Cần ít nhất một beam candidate")

    valid_indices = [index for index, item in enumerate(candidates) if item.caption]
    if not valid_indices:
        raise ValueError("Tất cả beam candidate đều rỗng")
    valid_captions = [candidates[index].caption for index in valid_indices]
    text_inputs = processor(
        text=valid_captions,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )
    text_inputs = {name: tensor.to(device) for name, tensor in text_inputs.items()}
    raw_text_features = encoder.get_text_features(**text_inputs)
    text_features = (
        raw_text_features.pooler_output
        if hasattr(raw_text_features, "pooler_output")
        else raw_text_features
    )

    valid_scores = compute_clip_similarity_scores(
        image_features.to(device),
        text_features,
    ).squeeze(1)
    all_scores = torch.full(
        (len(candidates),),
        fill_value=-torch.inf,
        dtype=valid_scores.dtype,
        device=valid_scores.device,
    )
    all_scores[torch.tensor(valid_indices, device=valid_scores.device)] = valid_scores

    rescored_candidates = tuple(
        replace(candidate, clip_score=float(all_scores[index].detach().cpu()))
        for index, candidate in enumerate(candidates)
    )
    selected_index = int(all_scores.argmax().item())
    selected = rescored_candidates[selected_index]
    return GenerationResult(
        caption=selected.caption,
        selected_beam_rank=selected.beam_rank,
        candidates=rescored_candidates,
    )

## 10. Smoke test cho beam search và cosine similarity

Ô này dùng GPT-2 rất nhỏ để kiểm tra beam search trả đúng năm sequence. Phần thứ hai dùng các vector chủ động thiết kế để xác nhận cosine reranking chọn đúng text feature gần image feature nhất và không sửa image feature gốc. Đây là kiểm tra logic, không phải kiểm tra chất lượng caption.

In [ ]:
class TinyTokenizer:
    eos_token_id = 0
    pad_token_id = 0

    @staticmethod
    def batch_decode(sequences: torch.Tensor, skip_special_tokens: bool = True) -> list[str]:
        captions = []
        for sequence in sequences.detach().cpu().tolist():
            kept_ids = [
                token_id
                for token_id in sequence
                if not skip_special_tokens or token_id != 0
            ]
            captions.append(" ".join(str(token_id) for token_id in kept_ids))
        return captions


tiny_gpt2 = GPT2LMHeadModel(
    GPT2Config(
        vocab_size=8,
        n_positions=8,
        n_ctx=8,
        n_embd=8,
        n_layer=1,
        n_head=1,
        bos_token_id=1,
        eos_token_id=0,
        pad_token_id=0,
    )
)
for parameter in tiny_gpt2.parameters():
    parameter.data.zero_()
tiny_gpt2.eval()

smoke_candidates = beam_search_from_prefix(
    tiny_gpt2,
    TinyTokenizer(),
    torch.zeros(1, 2, 8),
    max_new_tokens=4,
    num_beams=5,
    num_return_sequences=5,
)
assert len(smoke_candidates) == 5
assert [item.beam_rank for item in smoke_candidates] == [1, 2, 3, 4, 5]
assert all(torch.isfinite(torch.tensor(item.beam_score)) for item in smoke_candidates)

smoke_image_features = torch.tensor([[1.0, 0.0]])
original_image_features = smoke_image_features.clone()
smoke_text_features = torch.tensor(
    [[0.0, 1.0], [1.0, 0.0], [-1.0, 0.0]],
)
smoke_clip_scores = compute_clip_similarity_scores(
    smoke_image_features,
    smoke_text_features,
).squeeze(1)
assert int(smoke_clip_scores.argmax()) == 1
assert torch.equal(smoke_image_features, original_image_features)
print("Beam-search and CLIP-similarity smoke tests passed")

## 11. Chạy inference và xem kết quả

Ô này sinh năm beam rồi rerank bằng CLIP. Bảng kết quả giúp phân biệt hai tín hiệu: `beam_score` đo xác suất chuỗi theo GPT-2, còn `clip_score` đo độ gần semantic giữa caption và ảnh. Dòng `selected = True` là caption cuối cùng.

In [ ]:
beam_candidates = beam_search_from_prefix(
    clipcap_model.gpt2,
    tokenizer,
    visual_prefix,
    max_new_tokens=MAX_NEW_TOKENS,
    num_beams=BEAM_SIZE,
    num_return_sequences=NUM_RETURN_SEQUENCES,
    length_penalty=LENGTH_PENALTY,
    early_stopping=EARLY_STOPPING,
)

generation_result = clip_rerank_candidates(
    beam_candidates,
    image_features,
    clip_processor,
    clip_encoder,
    DEVICE,
)

candidate_table = pd.DataFrame(
    [
        {
            "beam_rank": item.beam_rank,
            "caption": item.caption,
            "beam_score": item.beam_score,
            "clip_score": item.clip_score,
            "selected": item.beam_rank == generation_result.selected_beam_rank,
        }
        for item in generation_result.candidates
    ]
)
display(candidate_table)
print(f"Final caption: {generation_result.caption}")
print(f"Selected beam rank: {generation_result.selected_beam_rank}")

## 12. Đóng gói thành hàm end-to-end

Sau khi đã kiểm tra từng tensor riêng lẻ, hàm dưới đây nối CLIP image encoding, Mapper, beam search và CLIP reranking lại với nhau. Đây là API gần nhất với phiên bản `.py` sau này, nhưng vẫn nhận các component qua tham số để dễ test và tránh phụ thuộc biến toàn cục.

In [ ]:
@torch.inference_mode()
def generate_caption(
    image_path: str | Path,
    processor: CLIPProcessor,
    encoder: CLIPModel,
    model: ClipCaptionModel,
    text_tokenizer: Any,
    device: torch.device,
    max_new_tokens: int = 15,
    num_beams: int = 5,
    num_return_sequences: int = 5,
    length_penalty: float = 1.0,
    early_stopping: bool = True,
) -> GenerationResult:
    selected_image = load_rgb_image(image_path)
    selected_features = encode_image_with_clip(
        selected_image,
        processor,
        encoder,
        device,
    )
    selected_prefix = build_visual_prefix(selected_features, model)
    candidates = beam_search_from_prefix(
        model.gpt2,
        text_tokenizer,
        selected_prefix,
        max_new_tokens=max_new_tokens,
        num_beams=num_beams,
        num_return_sequences=num_return_sequences,
        length_penalty=length_penalty,
        early_stopping=early_stopping,
    )
    return clip_rerank_candidates(
        candidates,
        selected_features,
        processor,
        encoder,
        device,
    )


final_result = generate_caption(
    IMAGE_PATH,
    clip_processor,
    clip_encoder,
    clipcap_model,
    tokenizer,
    DEVICE,
    max_new_tokens=MAX_NEW_TOKENS,
    num_beams=BEAM_SIZE,
    num_return_sequences=NUM_RETURN_SEQUENCES,
    length_penalty=LENGTH_PENALTY,
    early_stopping=EARLY_STOPPING,
)
print(f"Final caption: {final_result.caption}")

## 13. Training và inference khác nhau ở đâu?

Trong training, model nhận ground-truth `input_ids`, tạo text embedding, ghép sau visual prefix và dùng labels để tính loss. GPT-2 nhìn caption đúng theo teacher forcing.

Trong inference, không có ground-truth caption, không có labels và không có hard prompt. Visual prefix trực tiếp điều kiện hóa token đầu. Beam search giữ năm chuỗi có xác suất tốt, sau đó CLIP reranking chọn chuỗi gần ảnh nhất về semantic.

Thiết kế này giữ đúng conditioning của ClipCap khi training, đồng thời đồng nhất với ZeroCap ở phần so sánh quan trọng: năm beam, cùng ngân sách token và CLIP chọn một caption cuối.

Các lỗi thường gặp:

- Dùng CLIP model khác model đã trích feature khi training.
- Chuẩn hóa CLIP feature trước Mapper dù training dùng feature gốc. Chỉ bản sao dùng reranking mới được chuẩn hóa.
- Dựng Mapper khác cấu hình checkpoint.
- Dùng `best.pt` thay cho `final.pt` khi giao thức đánh giá yêu cầu cùng số epoch.
- Chèn hard prompt hoặc BOS dù training không dùng chúng.
- Quên `eval()`, khiến dropout làm kết quả thay đổi.
- Không dùng EOS hoặc không giới hạn số token.
- Nhầm beam score với CLIP score hoặc chọn beam đầu thay vì rerank.
- Dùng CLIPScore làm metric đánh giá duy nhất dù CLIP đã tham gia chọn caption.